# Problem Set 5: Regularize a Residential Sales-Price Model

You are advising analysts who need to predict a residential building project's
eventual sales price at the **start of construction**. You will compare an
unregularized linear model with tuned ridge and LASSO models, then assess their
predictive performance and coefficient sparsity.

## Required work and submission

- Use `data/residential_building.csv`.
- Complete Tasks 1–6 in code and show every requested table, scalar, or plot.
- Complete every written response in your own words, including Task 7.
- Use the fixed features, split, folds, grids, units, metric definitions, and
  decision rules stated below so answers are comparable.
- Keep the held-out test set untouched until Task 5.
- You may add or reorganize cells. Intermediate object names and code organization
  are your choice.
- Submit the completed `.ipynb` with visible outputs after restarting the kernel and
  running the notebook from top to bottom.

The target and model errors are measured in `10000 IRRm` (10,000 Iranian rials).
All 104 retained predictors are treated as numerical **as provided**,
including the coded locality variable `v_1`; do not one-hot encode it. Do not create
interactions or other artificial features.


## 0. Setup

Run the supplied imports, data-source helper, and constants. The helper first
searches for a local `data/residential_building.csv` and otherwise returns the
public course copy on GitHub, so Colab users do not need to upload the CSV. The
grids include the fixed candidates from 0.01 through 10,000. RMSE and MAE must be
reported in the target's `10000 IRRm` unit.


In [ ]:
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")

PUBLIC_REPOSITORY = "okuchap/GB656_2026_public"
PUBLIC_REVISION = "main"


def course_data_source(file_name):
    """Return a local course-data path when available, otherwise its public URL."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local_path = root / "data" / file_name
        if local_path.is_file():
            return local_path

    encoded_name = quote(file_name)
    return (
        "https://raw.githubusercontent.com/"
        f"{PUBLIC_REPOSITORY}/{PUBLIC_REVISION}/data/{encoded_name}"
    )


residential_source = course_data_source("residential_building.csv")

TEST_SIZE = 0.25
RANDOM_STATE = 606
N_SPLITS = 5
RIDGE_ALPHA_GRID = np.logspace(-2, 4, 13)
LASSO_ALPHA_GRID = np.logspace(-2, 4, 13)
NONZERO_TOLERANCE = 1e-8


## Task 1. Audit the prediction data

**Inputs and fixed assumptions**

- Load the project data with `pd.read_csv(residential_source)`; the course file is
  `data/residential_building.csv`.
- Target: `actual_sales_price`, measured in `10000 IRRm`
- Exclude `actual_construction_cost` (realized construction cost),
  `completion_year`, `completion_quarter`, and `v_7` (realized construction
  duration); each is known only after construction starts.
- Retain `v_8`: it is unit sales price per square meter recorded at project start.
- Treat every retained predictor as numerical as provided, including coded locality
  `v_1`.

**Required visible outputs**

1. Project-table shape and total number of missing project-table cells.
2. A direct numerical summary of `actual_sales_price`.
3. The candidate-predictor count after applying the exclusions; it must be 104.


In [ ]:
# TODO
# Add or reorganize cells as needed.


**Your response:** In 4–6 sentences, state the target and unit, explain every
prediction-time exclusion, explain why `v_8` remains, and explain why having many predictors relative to the number of projects, together with related lagged indicators, makes regularization potentially useful.


## Task 2. Explore the target and marginal relationships

Use only the 104 candidate predictors defined in Task 1.

**Required visible outputs**

1. A clearly labeled histogram of `actual_sales_price` with its unit.
2. A ten-row correlation screen ranked by absolute Pearson correlation with the
   target while preserving each signed correlation. Show feature name, signed
   correlation, and absolute correlation.
3. A clearly labeled scatterplot of `actual_sales_price` against `v_2` (total floor
   area in square meters).


In [ ]:
# TODO
# Add or reorganize cells as needed.


**Your response:** In 3–5 sentences, describe the target distribution, identify
notable marginal relationships from the correlation screen, explain why the strong
correlation for `v_8` is plausible, and explain why marginal correlations alone do
not choose the best multivariable prediction workflow.


## Task 3. Establish the evaluation boundary and unregularized baseline

Use all 104 candidate predictors and the target from Task 1.

**Fixed assumptions**

- Hold out 25% with `random_state=606`.
- Use five shuffled `KFold` splits with `random_state=606` on the training set only.
- Fit `LinearRegression` after `StandardScaler` in one pipeline.
- Calculate mean CV-RMSE by requesting `neg_root_mean_squared_error`, reversing its
  sign, and averaging the five fold RMSE values.
- Do not inspect test outcomes or calculate any test metric yet.

**Required visible outputs**

- Training and test feature-table dimensions.
- Baseline training RMSE and mean five-fold CV-RMSE.


In [ ]:
# TODO
# Add or reorganize cells as needed.


**Your response:** In 2–4 sentences, compare training RMSE with mean CV-RMSE and
explain why training error is commonly lower.


## Task 4. Tune ridge and LASSO on the training set

Create separate `StandardScaler`-plus-model pipelines for ridge and LASSO. Use
`Lasso(max_iter=200000, tol=0.001)`. Tune `model__alpha` with the supplied model-
specific grids, the Task 3 folds, `scoring="neg_root_mean_squared_error"`, and
`n_jobs=1`. Fit both searches using training data only.

**Required visible output**

For ridge and LASSO separately, print the selected alpha, positive mean CV-RMSE,
and whether the selected alpha is at that model's grid endpoint. Report these as
labeled scalar values rather than a pandas DataFrame.


In [ ]:
# TODO
# Add or reorganize cells as needed.


**Your response:** In 3–5 sentences, explain why scaling matters for the penalties,
why scaling must occur inside the cross-validation pipeline, and what you would do if
a selected alpha were at a grid endpoint.


## Task 5. Make the final held-out comparison

All tuning choices must now be frozen. Use the test set for the first and only model-
performance comparison among unregularized regression, tuned ridge, and tuned LASSO.

**Required visible output**

For each model, print a labeled result block containing training RMSE, mean CV-RMSE,
test RMSE, and test MAE. Use the baseline CV score from Task 3 and each tuned
search's selected CV score. Report the metrics as scalar values rather than a
pandas DataFrame.


In [ ]:
# TODO
# Add or reorganize cells as needed.


**Your response:** In 4–6 sentences, identify the lowest mean CV-RMSE and test RMSE,
state whether test RMSE and MAE rank the models the same way, compare the size of the
differences, and explain why one test split does not settle every future comparison.


## Task 6. Compare ridge and LASSO coefficients

For each selected regularized pipeline, recover the feature names and coefficients from the fitted pipeline, keeping them in their fitted order. A coefficient is nonzero when
`abs(coefficient) > NONZERO_TOLERANCE`.

**Required visible outputs**

For each model, print a labeled result block containing:

1. Total, nonzero, and zero coefficient counts.
2. A ranked list of the ten largest absolute coefficients, showing each feature
   name and signed coefficient.
3. The feature name and signed coefficient for the largest positive and largest
   negative coefficient.

Report these results without constructing a pandas DataFrame.


In [ ]:
# TODO
# Add or reorganize cells as needed.


**Your response:** Compare the ridge and LASSO coefficient counts and patterns.
Identify at least two features that appear among the largest absolute coefficients,
report their signs, and compare their relative magnitudes across the two models. Then
explain why LASSO can produce zeros while ridge usually does not, one way sparsity
could help an analyst, and one reason to treat the selected feature set cautiously.


## Task 7. Recommend a predictive workflow

**Your response:** In 5–8 sentences, recommend one of the three fitted workflows for
this start-of-construction prediction task. Support the recommendation with CV, test,
and sparsity results; state a realistic analyst use; mention the role of the strong
available signal `v_8`; and identify important evaluation or stability limitations.
